# 50 — ATS Rule Design
**Goal:** Design a transparent rule-based ATS scoring system.

Every ATS score should be a story: this resume covers 35% of the required skills, has five years against a three-year bar, and is missing two standard sections. This chapter builds the **rule set** that turns raw resume text into seven interpretable 0–100 scores, each with a fixed, explicit weight. Nothing here is a black box — every point is produced by a function you can read, test, and defend in a meeting.

**Why it matters for resumes / ATS:** recruiters and candidates both distrust an unexplained number. A transparent rule set is the difference between "the system says 64" and "the system says 64 because you matched 4/4 required skills but missed the summary section." Rules are also deterministic and cheap — the same resume always gets the same score — which makes them the foundation for everything else in this block: explanations, simulation, and ranking.

## 1. ATS Scoring Categories

Scoring is **decomposed**: instead of one opaque number, the ATS measures seven independent dimensions, each normalized to 0–100, then combines them with a **weighted sum**. The weights encode business priorities — skill match is worth seven times the boolean keyword check.

| Dimension | What it measures | Weight |
|---|---|---|
| `skill_match` | % of required JD skills found in resume | 35% |
| `experience` | years of experience vs. requirement | 20% |
| `education` | degree level match | 15% |
| `bullet_quality` | STAR compliance of experience bullets | 10% |
| `format` | parseability, section completeness | 10% |
| `duration` | employment stability, gaps | 5% |
| `boolean` | must-have keywords present | 5% |

**What the code does:** prints this dimension table, then instantiates `ATSScorer()` and confirms the weights sum to **1.0** — a sanity check, because if the weights ever drift off 1.0 the final 0–100 scale silently breaks. Note the header's "0-100 each": every dimension must be normalized *before* weighting, or a 200-character resume would be dwarfed by a 50,000-character one.

In [ ]:
print('''ATS scoring dimensions (0-100 each):
1. SKILL_MATCH:   % of required skills found in resume (weight: 35%)
2. EXPERIENCE:    Years of experience vs requirement (weight: 20%)
3. EDUCATION:     Degree level match (weight: 15%)
4. BULLET_QUALITY: STAR compliance of experience bullets (weight: 10%)
5. FORMAT:        Parseability, section completeness (weight: 10%)
6. DURATION:      Employment stability, gaps (weight: 5%)
7. BOOLEAN:       Must-have keywords present (weight: 5%)

Total = weighted sum of all dimensions (0-100).''')

## 2. Defining Scoring Rules

Each dimension gets its own **scoring function** — a small, readable rule instead of a model. `ATSScorer` stores the weights in a dict and implements four of the seven rules; the remaining three (`education`, `bullet_quality`, `duration`) are plugged in later.

**What the code does:**
- `skill_match_score()` — case-insensitive overlap of resume skills against JD skills, returned as `matched / len(jd_skills) * 100`; an empty JD list returns 0.
- `experience_score()` — ratio of resume years to required years mapped to **bands**: ≥1.5× → 100, ≥1.0× → 90, ≥0.75× → 70, ≥0.5× → 50, else 30; no requirement at all defaults to 80. Meeting the bar exactly gives 90, not 100 — the top band is reserved for candidates who *exceed* it.
- `format_score()` — starts at 100 and subtracts 15 per missing section keyword (`summary`, `experience`, `education`, `skills`), 30 for text under 200 chars, 20 for text over 50,000; floors at 0.
- `boolean_check()` — fraction of must-have terms found, again as a percentage.

**Try it:** run the cell and check `Weight sum: 1.0`. One trap to note: the section regexes are written `r"\\b"` — in a raw string that is an escaped backslash plus `b` (a literal `\b` pattern), not a word boundary — so in the Ch. 51 run every section is reported missing. The intended pattern is `r"\b"`.

In [ ]:
import re

class ATSScorer:
    def __init__(self):
        self.weights = {
            "skill_match": 0.35,
            "experience": 0.20,
            "education": 0.15,
            "bullet_quality": 0.10,
            "format": 0.10,
            "duration": 0.05,
            "boolean": 0.05,
        }
    
    def skill_match_score(self, resume_skills, jd_skills):
        if not jd_skills: return 0
        matched = sum(1 for s in jd_skills if s.lower() in [rs.lower() for rs in resume_skills])
        return (matched / len(jd_skills)) * 100
    
    def experience_score(self, resume_years, required_years):
        if not required_years: return 80
        ratio = resume_years / max(required_years, 1)
        if ratio >= 1.5: return 100
        if ratio >= 1.0: return 90
        if ratio >= 0.75: return 70
        if ratio >= 0.5: return 50
        return 30

    def format_score(self, resume_text):
        score = 100
        sections = ["summary", "experience", "education", "skills"]
        for s in sections:
            if not re.search(r"\\b" + s + r"\\b", resume_text, re.IGNORECASE):
                score -= 15
        if len(resume_text) < 200: score -= 30
        if len(resume_text) > 50000: score -= 20
        return max(0, score)

    def boolean_check(self, resume_text, must_have_terms):
        if not must_have_terms: return 100
        found = sum(1 for t in must_have_terms if t.lower() in resume_text.lower())
        return (found / len(must_have_terms)) * 100

scorer = ATSScorer()
print(f"ATS weights: {scorer.weights}")
print(f"Weight sum: {sum(scorer.weights.values())}")

## Summary: ATS scoring decomposes match quality into interpretable dimensions with transparent weights.

**Rules before models: a transparent weighted sum is the baseline every smarter scorer must beat.**

Seven normalized dimensions × fixed weights = one 0–100 score, with every point attributable to a specific rule. That property — *decomposability* — is what makes the rest of this block possible: you can explain a score only if you know which dimension produced it, and you can simulate or rank only if the scoring is repeatable.

This chapter feeds directly into Ch. 51, where each dimension score is paired with a human-readable reason, turning this weight table into a full explanation engine.